# SetWise — Multi-Dataset IMU Exercise Recognition v3

**Primary task**: Rep counting
**Auxiliary task**: Exercise classification (weight 0.1)

**Architecture**: Single-branch time-normalized (CNN-Attention-TCN)
- All inputs resampled to T_NORM=512 samples — rep count encoded as cycle frequency
- ConvStem (EEGNet-style) → Multi-head self-attention → Dilated causal TCN×2 → GlobalAvgPool

**Two-phase training**:
1. Pretrain on recofit + MyoGym (forearm) — learn general exercise representations
2. Fine-tune on wrist data (Whales1and2 + INSIGHT-LME + mm-fit) — adapt to wrist placement

**Key dataset decisions**:
- INSIGHT-LME `has_reps=True`: per-rep time-normalised → cycle count in T_NORM window encodes true rep count
- mm-fit `has_reps=False`: 92% of sets are exactly 10 reps — too narrow for rep regression
- recofit: fixed sentinel bug (-1 used for non-exercise rows, was leaking into rep loss)
- Whales1and2 `has_reps=True`: rep counts 2–30, primary real-time wrist rep regression source
- recofit annotated `has_reps=True`: 1 011 in-taxonomy sets, range 1–61, median 20

In [5]:
import numpy as np
import pandas as pd
import os
import gc
import warnings
from collections import Counter

import scipy.io
from scipy.signal import resample as scipy_resample
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error

warnings.filterwarnings('ignore')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Device: cuda


## Configuration & Exercise Taxonomy

In [6]:
BASE        = "/Users/znorton/Desktop/school/csc491/0SetWise"
TARGET_HZ   = 50
T_NORM      = 512    # all inputs resampled to this length; rep count → cycle frequency
N_CHANNELS  = 6      # acc XYZ + gyro XYZ
MIN_SAMPLES = 40     # drop exercise classes with fewer total samples

REP_WEIGHT  = 1.0    # rep regression is primary
CLS_WEIGHT  = 0.1    # classification is auxiliary

# ── Exercise taxonomy ─────────────────────────────────────────────────────────
EXERCISE_MAP = {
    # ── INSIGHT-LME (wrist) ──────────────────────────────────────────────────
    'Bicep Curls':          'bicep_curls',
    'Lateral Raise':        'lateral_raises',
    'Lunges':               'lunges',
    'Squats':               'squats',
    'Triceps Extension':    'tricep_extensions',
    'Frontal Raise':        'frontal_raise',

    # ── mm-fit (wrist) ───────────────────────────────────────────────────────
    'squats':                   'squats',
    'lunges':                   'lunges',
    'bicep_curls':              'bicep_curls',
    'situps':                   'situps',
    'pushups':                  'pushups',
    'tricep_extensions':        'tricep_extensions',
    'dumbbell_rows':            'rows',
    'jumping_jacks':            'jumping_jacks',
    'dumbbell_shoulder_press':  'shoulder_press',
    'lateral_shoulder_raises':  'lateral_raises',

    # ── Whales1and2 (wrist) ──────────────────────────────────────────────────
    'APULL':  'pullups',
    'CGCR':   'rows',
    'CGOCTE': 'tricep_extensions',
    'DLR':    'lateral_raises',
    'DSP':    'shoulder_press',
    'IDBC':   'bicep_curls',
    'AIDBC':  'bicep_curls',
    'MGTBR':  'rows',
    'MIBP':   'bench_press',
    'MSP':    'shoulder_press',
    'MTE':    'tricep_extensions',
    'NGCR':   'rows',
    'PREC':   'bicep_curls',
    'SACLR':  'lateral_raises',
    'SAOCTE': 'tricep_extensions',
    'SAODTE': 'tricep_extensions',
    '30BP':   'bench_press',
    '30DBP':  'bench_press',
    '45DBP':  'bench_press',

    # ── recofit (forearm — pretrain only) ────────────────────────────────────
    'Bicep Curl':                                           'bicep_curls',
    'Biceps Curl (band)':                                   'bicep_curls',
    'Two-arm Dumbbell Curl (both arms, not alternating)':   'bicep_curls',
    'Alternating Dumbbell Curl':                            'bicep_curls',
    'Lateral Raise':                                        'lateral_raises',
    'Shoulder Press (dumbbell)':                            'shoulder_press',
    'Squat Rack Shoulder Press':                            'shoulder_press',
    'Overhead Triceps Extension':                           'tricep_extensions',
    'Overhead Triceps Extension (label spans both arms)':   'tricep_extensions',
    'Triceps Kickback (knee on bench) (label spans both arms)': 'tricep_extensions',
    'Triceps Kickback (knee on bench) (left arm)':          'tricep_extensions',
    'Triceps Kickback (knee on bench) (right arm)':         'tricep_extensions',
    'Triceps extension (lying down)':                       'tricep_extensions',
    'Triceps extension (lying down) (left arm)':            'tricep_extensions',
    'Triceps extension (lying down) (right arm)':           'tricep_extensions',
    'Squat':                                                'squats',
    'Squat (arms in front of body, parallel to ground)':    'squats',
    'Squat (hands behind head)':                            'squats',
    'Squat (kettlebell / goblet)':                          'squats',
    'Dumbbell Squat (hands at side)':                       'squats',
    'Squat Jump':                                           'squats',
    'Wall Squat':                                           'squats',
    'Lunge (alternating both legs, weight optional)':       'lunges',
    'Walking lunge':                                        'lunges',
    'Dumbbell Row (knee on bench) (label spans both arms)': 'rows',
    'Dumbbell Row (knee on bench) (left arm)':              'rows',
    'Dumbbell Row (knee on bench) (right arm)':             'rows',
    'Band Pull-Down Row':                                   'rows',
    'Dumbbell Deadlift Row':                                'rows',
    'Pushup (knee or foot variation)':                      'pushups',
    'Pushups':                                              'pushups',
    'Sit-up (hands positioned behind head)':                'situps',
    'Sit-ups':                                              'situps',
    'Butterfly Sit-up':                                     'situps',
    'Crunch':                                               'situps',
    'Jumping Jacks':                                        'jumping_jacks',
    'Chest Press (rack)':                                   'bench_press',
    'Burpee':                                               'burpee',
}

# ── MyoGym integer → canonical name ──────────────────────────────────────────
MYOGYM_LABEL_MAP = {
    1:  'rows',              2:  'rows',              3:  'rows',
    4:  'rows',              5:  'rows',              6:  'rows',
    7:  'rows',              8:  'bench_press',       9:  None,
    10: 'bench_press',       11: None,                12: 'pushups',
    13: 'bench_press',       14: 'bench_press',       15: 'tricep_extensions',
    16: 'tricep_extensions', 17: 'tricep_extensions', 18: 'tricep_extensions',
    19: 'tricep_extensions', 20: 'bicep_curls',       21: 'bicep_curls',
    22: 'bicep_curls',       23: 'bicep_curls',       24: 'bicep_curls',
    25: 'bicep_curls',       26: 'rows',              27: 'lateral_raises',
    28: 'frontal_raise',     29: 'shoulder_press',    30: None,
}

## Dataset Loaders

In [7]:
def _ds(arr, src_hz):
    return arr[:: src_hz // TARGET_HZ] if src_hz != TARGET_HZ else arr


def load_insight(base=BASE):
    """INSIGHT-LME, 100 Hz wrist. Each trial is time-normalised to 30 samples per rep.
    has_reps=True: after whole-set resampling to T_NORM, cycle count encodes true rep count."""
    df   = pd.read_csv(os.path.join(base, 'INSIGHT-LME', 'INSIGHT_LME_dataset.csv'))
    cols = ['Accelerometer_X','Accelerometer_Y','Accelerometer_Z',
            'Gyroscope_X','Gyroscope_Y','Gyroscope_Z']
    segs = []
    for (pid, ex), grp in df.groupby(['Participant_ID', 'Exercise_Label']):
        if ex not in EXERCISE_MAP: continue
        imu  = _ds(grp[cols].values.astype(np.float32), 100)
        segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex],
                     'reps': float(grp['Repetition_Count'].max()),
                     'source': 'insight', 'has_reps': True})
    print(f'INSIGHT-LME  : {len(segs):>5} segments')
    return segs


def load_mmfit(base=BASE):
    """mm-fit, 50 Hz left-wrist.
    has_reps=False: 92% of sets are exactly 10 reps — too narrow for rep regression."""
    mmdir = os.path.join(base, 'mm-fit')
    segs  = []
    for w in range(21):
        pdir = os.path.join(mmdir, f'w{w:02d}')
        lp   = os.path.join(pdir, f'w{w:02d}_labels.csv')
        ap   = os.path.join(pdir, f'w{w:02d}_sw_l_acc.npy')
        gp   = os.path.join(pdir, f'w{w:02d}_sw_l_gyr.npy')
        if not all(os.path.exists(p) for p in [lp, ap, gp]): continue
        labels = pd.read_csv(lp, header=None,
                             names=['start_frame','end_frame','reps','exercise_name'])
        acc, gyr = np.load(ap), np.load(gp)
        for _, row in labels.iterrows():
            ex = str(row['exercise_name'])
            if ex not in EXERCISE_MAP: continue
            s, e = int(row['start_frame']), int(row['end_frame'])
            imu  = np.concatenate([acc[s:e, 2:5], gyr[s:e, 2:5]], axis=1).astype(np.float32)
            if len(imu) < 20: continue
            segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex],
                         'reps': np.nan, 'source': 'mmfit', 'has_reps': False})
    print(f'mm-fit       : {len(segs):>5} segments')
    return segs


def load_whales(base=BASE):
    """Whales1and2, 100 Hz wrist. Rep counts 2–30; primary real-time rep regression source."""
    wdir     = os.path.join(base, 'Whales1and2')
    acc_cols = ['wristMotion_accelerationX','wristMotion_accelerationY','wristMotion_accelerationZ']
    gyr_cols = ['wristMotion_rotationRateX', 'wristMotion_rotationRateY', 'wristMotion_rotationRateZ']
    segs     = []
    for fname in sorted(os.listdir(wdir)):
        if not fname.endswith('.csv'): continue
        df = pd.read_csv(os.path.join(wdir, fname)).dropna(subset=acc_cols + gyr_cols)
        if len(df) < 50: continue
        ex = str(df['activity'].iloc[0])
        if ex not in EXERCISE_MAP: continue
        imu  = _ds(df[acc_cols + gyr_cols].values.astype(np.float32), 100)
        segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex],
                     'reps': float(df['reps'].iloc[0]),
                     'source': 'whales', 'has_reps': True})
    print(f'Whales1and2  : {len(segs):>5} segments')
    return segs


def load_recofit(base=BASE):
    """recofit, 50 Hz forearm arm-band (PRETRAIN ONLY). 1 011 annotated in-taxonomy sets,
    rep range 1–61. Bug fix: recofit uses -1 as sentinel for non-exercise rows; guard r > 0."""
    path = os.path.join(base, 'recofit', 'exercise_data.50.0000_singleonly.mat')
    segs = []
    print('Loading recofit (~3 min)...')
    m          = scipy.io.loadmat(path, struct_as_record=False, squeeze_me=True)
    activities = list(m['exerciseConstants'].activities)
    sd         = m['subject_data']
    for i in range(sd.shape[0]):
        for j in range(sd.shape[1]):
            cell = sd[i, j]
            if isinstance(cell, (float, np.floating)) and cell == 0: continue
            ex_name = activities[j]
            if ex_name not in EXERCISE_MAP: continue
            recs = [cell] if not hasattr(cell, '__len__') else cell
            for rec in recs:
                try:
                    imu  = np.concatenate([
                        rec.data.accelDataMatrix[:, 1:4],
                        rec.data.gyroDataMatrix[:,  1:4]
                    ], axis=1).astype(np.float32)
                    reps = float(getattr(rec, 'activityReps', float('nan')))
                    if len(imu) < 20: continue
                    # -1 is recofit's sentinel for non-exercise / unannotated rows
                    has_reps = not np.isnan(reps) and reps > 0
                    segs.append({'imu': imu, 'exercise': EXERCISE_MAP[ex_name],
                                 'reps': reps if has_reps else np.nan,
                                 'source': 'recofit', 'has_reps': has_reps})
                except Exception:
                    continue
    del m, sd; gc.collect()
    print(f'recofit      : {len(segs):>5} segments')
    return segs


def load_myogym(base=BASE):
    """MyoGym, ~60 Hz Myo Armband forearm — PRETRAIN ONLY, no rep annotations."""
    path = os.path.join(base, 'MyoGym', 'data', 'MyoGym', 'MyoGym.mat')
    print('Loading MyoGym...')
    m         = scipy.io.loadmat(path, struct_as_record=False, squeeze_me=True)
    raw       = m['raw_data']
    label_col = m['raw_data_labels'][:, 0].astype(int)
    del m; gc.collect()

    boundaries = np.concatenate([[0],
                                  np.where(np.diff(label_col) != 0)[0] + 1,
                                  [len(label_col)]])
    segs = []
    for k in range(len(boundaries) - 1):
        start, end = boundaries[k], boundaries[k + 1]
        lbl = label_col[start]
        if lbl not in MYOGYM_LABEL_MAP: continue
        ex_name = MYOGYM_LABEL_MAP[lbl]
        if ex_name is None: continue
        imu = raw[start:end, [10, 11, 12, 14, 15, 16]].astype(np.float32)
        if len(imu) < 50: continue
        segs.append({'imu': imu, 'exercise': ex_name,
                     'reps': np.nan, 'source': 'myogym', 'has_reps': False})

    print(f'MyoGym       : {len(segs):>5} segments')
    return segs

## Load All Datasets

Wrist sensors (Phase 2 fine-tuning): INSIGHT-LME, mm-fit, Whales1and2
Forearm sensors (Phase 1 pretraining only): recofit, MyoGym

In [8]:
wrist_segments    = load_insight() + load_mmfit() + load_whales()
recofit_segments  = load_recofit()
myogym_segments   = load_myogym()
pretrain_segments = recofit_segments + myogym_segments
all_segments      = wrist_segments + pretrain_segments

print(f'\nWrist: {len(wrist_segments)}   '
      f'Recofit: {len(recofit_segments)}   '
      f'MyoGym: {len(myogym_segments)}   '
      f'Total: {len(all_segments)}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/znorton/Desktop/school/csc491/0SetWise/INSIGHT-LME/INSIGHT_LME_dataset.csv'

## Exercise Frequency Analysis & Filtering

Keep only classes with ≥ MIN_SAMPLES across the combined dataset.

In [ ]:
ex_counts = Counter(s['exercise'] for s in all_segments)

print('Exercise distribution (combined, before filtering):')
for ex, cnt in sorted(ex_counts.items(), key=lambda x: -x[1]):
    wrist_cnt    = sum(1 for s in wrist_segments    if s['exercise'] == ex)
    pretrain_cnt = sum(1 for s in pretrain_segments if s['exercise'] == ex)
    flag = '  <- DROP' if cnt < MIN_SAMPLES else ''
    print(f'  {ex:<32} {cnt:>5}  (wrist {wrist_cnt:>4} | pretrain {pretrain_cnt:>4}){flag}')

keep_exercises    = {ex for ex, cnt in ex_counts.items() if cnt >= MIN_SAMPLES}
wrist_segments    = [s for s in wrist_segments    if s['exercise'] in keep_exercises]
pretrain_segments = [s for s in pretrain_segments if s['exercise'] in keep_exercises]

print(f'\nKept {len(keep_exercises)} classes: {sorted(keep_exercises)}')
print(f'Wrist: {len(wrist_segments)}  Pretrain: {len(pretrain_segments)}')

## Preprocessing

- All inputs scaled then resampled to T_NORM=512 — rep count becomes cycle frequency.
- Scaler fit on raw wrist train timesteps (before resampling) to preserve true signal statistics.
- `has_reps=False` segments (mm-fit: 10-rep bias; MyoGym: unannotated) excluded from rep loss.
- recofit -1 sentinel filtered: only `reps > 0` contributes to rep regression.

In [ ]:
le          = LabelEncoder().fit(sorted(keep_exercises))
label_names = le.classes_
print('Classes:', label_names)

def encode_reps(segs):
    """Fill NaN reps with median of annotated values; NaN entries never hit the loss."""
    r      = np.array([s['reps'] for s in segs], dtype=np.float32)
    median = float(np.nanmedian(r[~np.isnan(r)])) if np.any(~np.isnan(r)) else 10.0
    return np.where(np.isnan(r), median, r)

# ── Wrist: stratified 70/15/15 split ─────────────────────────────────────────
yw   = le.transform([s['exercise'] for s in wrist_segments]).astype(np.int64)
idxw = np.arange(len(yw))
idxw_tv, idxw_test = train_test_split(idxw, test_size=0.15, random_state=42, stratify=yw)
idxw_tr, idxw_val  = train_test_split(idxw_tv, test_size=0.15/0.85, random_state=42,
                                       stratify=yw[idxw_tv])
print(f'Wrist split — train: {len(idxw_tr)}, val: {len(idxw_val)}, test: {len(idxw_test)}')

# ── Scaler: fit on raw wrist train timesteps (before resampling) ──────────────
flat_wrist_tr = np.concatenate([wrist_segments[i]['imu'] for i in idxw_tr])
scaler        = StandardScaler().fit(flat_wrist_tr)

def prep_norm(segs, idxs):
    """Scale then resample each segment to T_NORM. Returns (N, T_NORM, 6)."""
    X = []
    for i in idxs:
        imu = scaler.transform(segs[i]['imu']).astype(np.float32)
        imu = scipy_resample(imu, T_NORM).astype(np.float32)
        X.append(imu)
    return np.stack(X)

Xw_tr  = prep_norm(wrist_segments, idxw_tr)
Xw_val = prep_norm(wrist_segments, idxw_val)
Xw_te  = prep_norm(wrist_segments, idxw_test)
rw_tr  = encode_reps([wrist_segments[i] for i in idxw_tr])
rw_val = encode_reps([wrist_segments[i] for i in idxw_val])
rw_te  = encode_reps([wrist_segments[i] for i in idxw_test])
yw_tr, yw_val, yw_te = yw[idxw_tr], yw[idxw_val], yw[idxw_test]

hrw_tr  = np.array([wrist_segments[i]['has_reps'] for i in idxw_tr],  dtype=bool)
hrw_val = np.array([wrist_segments[i]['has_reps'] for i in idxw_val], dtype=bool)
hrw_te  = np.array([wrist_segments[i]['has_reps'] for i in idxw_test], dtype=bool)
print(f'Wrist train — rep-annotated: {hrw_tr.sum()}/{len(hrw_tr)}  '
      f'val: {hrw_val.sum()}/{len(hrw_val)}  '
      f'test: {hrw_te.sum()}/{len(hrw_te)}')

# ── Pretrain (recofit + MyoGym): 85/15 split ─────────────────────────────────
yp   = le.transform([s['exercise'] for s in pretrain_segments]).astype(np.int64)
idxp = np.arange(len(yp))
idxp_tr, idxp_val = train_test_split(idxp, test_size=0.15, random_state=42, stratify=yp)
print(f'Pretrain split — train: {len(idxp_tr)}, val: {len(idxp_val)}')

Xp_tr  = prep_norm(pretrain_segments, idxp_tr)
Xp_val = prep_norm(pretrain_segments, idxp_val)
rp_tr  = encode_reps([pretrain_segments[i] for i in idxp_tr])
rp_val = encode_reps([pretrain_segments[i] for i in idxp_val])
yp_tr, yp_val = yp[idxp_tr], yp[idxp_val]

hrp_tr  = np.array([pretrain_segments[i]['has_reps'] for i in idxp_tr],  dtype=bool)
hrp_val = np.array([pretrain_segments[i]['has_reps'] for i in idxp_val], dtype=bool)
print(f'Pretrain train — rep-annotated: {hrp_tr.sum()}/{len(hrp_tr)}  '
      f'val: {hrp_val.sum()}/{len(hrp_val)}')

print(f'\nXw_tr: {Xw_tr.shape}')
print(f'Xp_tr: {Xp_tr.shape}')

## Datasets & DataLoaders

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, X, y, reps, has_reps=None):
        self.X        = torch.tensor(X.transpose(0, 2, 1), dtype=torch.float32)  # (N,6,T_NORM)
        self.y        = torch.tensor(y,    dtype=torch.long)
        self.reps     = torch.tensor(reps, dtype=torch.float32).unsqueeze(1)
        n = len(y)
        self.has_reps = torch.ones(n, dtype=torch.bool) if has_reps is None \
                        else torch.tensor(has_reps, dtype=torch.bool)
    def __len__(self):  return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i], self.reps[i], self.has_reps[i]


BATCH = 64

pre_train_loader = DataLoader(
    IMUDataset(Xp_tr,  yp_tr,  rp_tr,  hrp_tr),  BATCH, shuffle=True,  num_workers=0)
pre_val_loader   = DataLoader(
    IMUDataset(Xp_val, yp_val, rp_val, hrp_val),  BATCH, shuffle=False, num_workers=0)

ft_train_loader  = DataLoader(
    IMUDataset(Xw_tr,  yw_tr,  rw_tr,  hrw_tr),  BATCH, shuffle=True,  num_workers=0)
ft_val_loader    = DataLoader(
    IMUDataset(Xw_val, yw_val, rw_val, hrw_val),  BATCH, shuffle=False, num_workers=0)
ft_test_loader   = DataLoader(
    IMUDataset(Xw_te,  yw_te,  rw_te,  hrw_te),  BATCH, shuffle=False, num_workers=0)

print('Loaders ready.')

## Model — SetWiseV3

Single-branch time-normalised architecture:
```
Input (B, 6, T_NORM)
  ConvStem      EEGNet-style depthwise separable conv
  MHABlock      Pre-norm multi-head self-attention (no padding mask — fixed length)
  DilatedResBlock  dilation=1
  DilatedResBlock  dilation=2
  AdaptiveAvgPool1d(1)
  rep_head (primary)  / class_head (auxiliary, weight=0.1)
```

In [ ]:
class LayerNorm1d(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.norm = nn.LayerNorm(C)
    def forward(self, x):
        return self.norm(x.transpose(1, 2)).transpose(1, 2)


class ConvStem(nn.Module):
    def __init__(self, in_ch=6, F1=16, D=2, stem_kern=32, dropout=0.1):
        super().__init__()
        F2 = F1 * D
        self.temporal = nn.Sequential(
            nn.Conv1d(in_ch, F1, stem_kern, padding=stem_kern // 2, bias=False),
            LayerNorm1d(F1), nn.ELU(),
        )
        self.depthwise = nn.Sequential(
            nn.Conv1d(F1, F2, 1, groups=F1, bias=False),
            LayerNorm1d(F2), nn.ELU(), nn.Dropout(dropout),
        )
        self.pointwise = nn.Sequential(
            nn.Conv1d(F2, F2, 10, padding=5, bias=False),
            LayerNorm1d(F2), nn.ELU(), nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.pointwise(self.depthwise(self.temporal(x)))


class MHABlock(nn.Module):
    """Pre-norm MHA. No key_padding_mask needed — all inputs are T_NORM length."""
    def __init__(self, d_model, num_heads=4, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h, _ = self.attn(self.norm(x), self.norm(x), self.norm(x))
        return x + self.drop(h)


class DilatedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=4, dilation=1, dropout=0.3):
        super().__init__()
        p = (k - 1) * dilation
        self.pad1  = nn.ConstantPad1d((p, 0), 0)
        self.conv1 = nn.Conv1d(in_ch,  out_ch, k, dilation=dilation, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.pad2  = nn.ConstantPad1d((p, 0), 0)
        self.conv2 = nn.Conv1d(out_ch, out_ch, k, dilation=dilation, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Conv1d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.drop  = nn.Dropout(dropout)
        self.act   = nn.ELU()
    def forward(self, x):
        h = self.drop(self.act(self.bn1(self.conv1(self.pad1(x)))))
        h = self.drop(self.act(self.bn2(self.conv2(self.pad2(h)))))
        return self.act(h + self.skip(x))


class SetWiseV3(nn.Module):
    """Single-branch time-normalised model. Rep counting primary, classification auxiliary."""
    def __init__(self, in_channels=N_CHANNELS, num_classes=10,
                 F1=16, D=2, stem_kern=32,
                 num_heads=4, attn_dropout=0.1,
                 tcn_ch=64, tcn_k=4, tcn_dropout=0.3,
                 head_dropout=0.4):
        super().__init__()
        F2 = F1 * D
        self.stem = ConvStem(in_channels, F1=F1, D=D,
                             stem_kern=stem_kern, dropout=attn_dropout)
        self.mha  = MHABlock(F2, num_heads=num_heads, dropout=attn_dropout)
        self.tcn1 = DilatedResBlock(F2,     tcn_ch, k=tcn_k, dilation=1, dropout=tcn_dropout)
        self.tcn2 = DilatedResBlock(tcn_ch, tcn_ch, k=tcn_k, dilation=2, dropout=tcn_dropout)
        self.pool = nn.AdaptiveAvgPool1d(1)

        self.rep_head = nn.Sequential(
            nn.Linear(tcn_ch, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
        self.class_head = nn.Sequential(
            nn.Linear(tcn_ch, 64), nn.ReLU(), nn.Dropout(head_dropout),
            nn.Linear(64, num_classes)
        )

    def encode(self, x):
        h = self.stem(x)                              # (B, F2, T_NORM)
        h = self.mha(h.transpose(1, 2)).transpose(1, 2)  # (B, F2, T_NORM)
        h = self.tcn1(h)                              # (B, tcn_ch, T_NORM)
        h = self.tcn2(h)                              # (B, tcn_ch, T_NORM)
        return self.pool(h).squeeze(-1)               # (B, tcn_ch)

    def forward(self, x):
        h = self.encode(x)
        return self.class_head(h), self.rep_head(h)


model    = SetWiseV3(num_classes=len(label_names)).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## Training Utilities

In [ ]:
cls_loss_fn = nn.CrossEntropyLoss()
rep_loss_fn = nn.L1Loss()


def run_epoch(loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    ys, yhats, rs, rhats, hrs = [], [], [], [], []

    for xb, yb, rb, hrb in loader:
        xb, yb, rb = xb.to(device), yb.to(device), rb.to(device)
        hrb = hrb.to(device)
        with torch.set_grad_enabled(training):
            logits, rep_pred = model(xb)
            cls_loss = cls_loss_fn(logits, yb)
            if hrb.any():
                rep_loss = rep_loss_fn(rep_pred[hrb], rb[hrb])
                loss = REP_WEIGHT * rep_loss + CLS_WEIGHT * cls_loss
            else:
                loss = CLS_WEIGHT * cls_loss
            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * xb.size(0)
        ys.append(yb.cpu().numpy())
        yhats.append(logits.argmax(1).detach().cpu().numpy())
        rs.append(rb.cpu().numpy().ravel())
        rhats.append(rep_pred.detach().cpu().numpy().ravel())
        hrs.append(hrb.cpu().numpy())

    ys,   yhats = np.concatenate(ys),   np.concatenate(yhats)
    rs,   rhats = np.concatenate(rs),   np.concatenate(rhats)
    hrs         = np.concatenate(hrs)
    mae = mean_absolute_error(rs[hrs], rhats[hrs]) if hrs.any() else float('nan')

    return (total_loss / len(loader.dataset),
            (ys == yhats).mean(),
            mae,
            ys, yhats, rs, rhats)


def train_phase(train_loader, val_loader, epochs, lr, label, freeze_encoder=False):
    if freeze_encoder:
        for p in list(model.stem.parameters()) + \
                 list(model.mha.parameters())  + \
                 list(model.tcn1.parameters()) + \
                 list(model.tcn2.parameters()):
            p.requires_grad_(False)
        params = filter(lambda p: p.requires_grad, model.parameters())
        print(f'[{label}] Encoder frozen — training heads only')
    else:
        for p in model.parameters(): p.requires_grad_(True)
        params = model.parameters()
        print(f'[{label}] All parameters trainable')

    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr/100)

    best_loss, best_state = float('inf'), None
    for epoch in range(epochs):
        tr = run_epoch(train_loader, optimizer)
        vl = run_epoch(val_loader)
        scheduler.step()
        if vl[0] < best_loss:
            best_loss  = vl[0]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f'  [{label}] Ep {epoch+1:3d} | '
                  f'tr loss {tr[0]:.4f} rep-mae {tr[2]:.2f} acc {tr[1]:.3f} | '
                  f'vl loss {vl[0]:.4f} rep-mae {vl[2]:.2f} acc {vl[1]:.3f}')

    model.load_state_dict(best_state)
    print(f'  [{label}] Best val loss: {best_loss:.4f}\n')
    return best_state

## Phase 1 — Pretrain on recofit + MyoGym (forearm)

Rep head trains on recofit annotated sets (1 011 sets, range 1–61).
Cycle frequency in T_NORM input encodes rep count; transfers across sensor placements.

In [ ]:
print('=== Phase 1: Pretrain on recofit + MyoGym ===')
pretrain_state = train_phase(
    pre_train_loader, pre_val_loader,
    epochs=60, lr=1e-3,
    label='pretrain',
    freeze_encoder=False
)

## Phase 2a — Fine-tune heads only (frozen encoder)

In [ ]:
print('=== Phase 2a: Fine-tune heads (encoder frozen) ===')
train_phase(
    ft_train_loader, ft_val_loader,
    epochs=20, lr=1e-3,
    label='ft-heads',
    freeze_encoder=True
)

## Phase 2b — Fine-tune full model (lower LR)

In [ ]:
print('=== Phase 2b: Full fine-tune on wrist data ===')
train_phase(
    ft_train_loader, ft_val_loader,
    epochs=50, lr=2e-4,
    label='ft-full',
    freeze_encoder=False
)

## Evaluation on Wrist Test Set

In [ ]:
te_loss, te_acc, te_mae, y_true, y_pred, r_true, r_pred = run_epoch(ft_test_loader)

print(f'Test rep MAE   : {te_mae:.4f}  (primary)')
print(f'Off-by-1 acc   : {np.mean(np.abs(np.round(r_pred[hrw_te]) - r_true[hrw_te]) <= 1):.4f}')
print(f'Test accuracy  : {te_acc:.4f}  (auxiliary)')
print()
print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))

## Class Distribution in Test Set

In [ ]:
print('Test set distribution:')
for ex, cnt in sorted(Counter(label_names[i] for i in y_true).items(), key=lambda x: -x[1]):
    print(f'  {ex:<32} {cnt}')

## Example Inference

In [ ]:
model.eval()
print(f"{'#':>4}  {'true_exercise':<30} {'pred_exercise':<30}  true_reps  pred_reps")
print('-' * 90)
for i in range(min(15, len(yw_te))):
    xb = torch.tensor(Xw_te[i:i+1].transpose(0, 2, 1), dtype=torch.float32).to(device)
    with torch.no_grad():
        logits, rep_out = model(xb)
    pred_cls = logits.argmax(1).item()
    true_reps = f'{rw_te[i]:.1f}' if hrw_te[i] else 'N/A'
    print(f"{i:>4}  {label_names[yw_te[i]]:<30} {label_names[pred_cls]:<30}  "
          f"{true_reps:>9}  {rep_out.item():>9.2f}")